# Paper 26 backbone on YOLO26n: MSRA (C2PSA with Mona adapters)

| Part | Source |
|---|---|
| **Backbone (layers 0–10)** | **Paper 26:** the PSA block inside the C2PSA at layer 10 replaced by **2 Mona adapters** (MSRA); every other backbone layer is YOLO26n base |
| Neck FPN + PAN (layers 11–22) | YOLO26n base, unchanged (LSA of Paper 26 is not used) |
| Head (Detect, layer 23) | YOLO26n base, unchanged (one-to-one, NMS-free) |

The baseline ("Standard YOLO26n Backbone") is **not trained here**; its values are already filled in the final table.

**Kaggle settings:** Accelerator = **GPU T4 x1**, Internet = **On**, keep the notebook **Private**

## 1. Setup

In [ ]:
# ultralytics version is pinned: use the same version on Kaggle, Raspberry Pi 5 and Jetson
!pip install -q ultralytics==8.4.137 roboflow

In [ ]:
import os, sys, gc, shutil, random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch, torchvision
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (turn on the GPU accelerator)")

WORK = Path("/kaggle/working")
os.chdir(WORK)
sys.path.insert(0, str(WORK))  # so msra_backbone.py can be imported

## 2. Dataset (download from Roboflow)

The Roboflow API key is written in the cell below. Keep this notebook **Private** on Kaggle.

In [ ]:
import os, yaml
from roboflow import Roboflow

os.chdir("/kaggle/working")

# Roboflow API key (keep this notebook Private on Kaggle)
API_KEY = "CxHgc1Nn3FK5vYNsBtJW"

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("md-mahibullah").project("digitalaccess-2-copy")
dataset = project.version(4).download("yolo26")  # Roboflow YOLO26 export (same label format as YOLOv8)

# Rename and fix paths
new = "/kaggle/working/digitalaccess"
if os.path.exists(dataset.location) and not os.path.exists(new):
    os.rename(dataset.location, new)

# Update YAML
yaml_path = f"{new}/data.yaml"
with open(yaml_path) as f:
    data = yaml.safe_load(f)
data["path"], data["train"], data["val"] = new, "train/images", "valid/images"
if os.path.exists(f"{new}/test/images"):
    data["test"] = "test/images"
else:
    data.pop("test", None)
with open(yaml_path, "w") as f:
    yaml.dump(data, f)

# the following cells use these variables
DATA_ROOT = Path(new)
DATA_YAML = yaml_path
names = data["names"]
assert data.get("test"), "No test split: the evaluation and FPS steps need a test split"

for s in ["train", "valid", "test"]:
    print(f"{s:6s}: {len(list((DATA_ROOT / s / 'images').glob('*')))} images")
print("nc =", len(names))
print("✅ Dataset prepared!")

## 3. Paper 26 backbone: MSRA (Multi-Scale Refinement Adapter)

**What MSRA does** (Paper 26, Eq. 10–15). It keeps the C2PSA layout at layer 10 (after SPPF):
1. 1×1 conv (`cv1`) → split into a *stable* branch Ra and a *refinement* branch Rb.
2. Ra → 1×1 projection (R̄a).
3. Rb → **2 stacked Mona adapters** (instead of the PSA self-attention block).
4. concat(R̄a, Rb) → 1×1 conv (`cv2`).

**One Mona adapter:**
- R̂ = LayerNorm(R) · γ + R · γx (learnable per-channel γ, γx);
- P1 = 1×1 projection to the adapter width (64 channels);
- Dm = (DW 3×3 + DW 5×5 + DW 7×7)(P1) / 3 + P1;
- Do = Dm + 1×1(Dm);  Da = Dropout(GELU(Do));  P2 = 1×1 back to the branch width;
- output = R + P2.

**Where it sits in YOLO26n.**

| Layer | Base YOLO26n | Paper 26 backbone | Output |
|---|---|---|---|
| 10 | C2PSA (cv1 → PSA block → cv2) | **MSRA** (cv1 → 1×1 on Ra, 2 Mona adapters on Rb → cv2) | P5/32 (20 × 20, 256 ch) |

Input and output channels stay the same, so no layer index changes and the neck connections stay the same.

**Pretrained weights (stable start).** `yolo26n.pt` is loaded first, then layer 10 is changed:
- `cv1` and `cv2` keep the pretrained C2PSA weights (same shapes);
- the new 1×1 projection on Ra starts as an identity;
- the last 1×1 conv of each Mona adapter starts at zero, so each adapter starts as an identity.

Only the PSA block is dropped (the Mona adapters are new).

**Not taken from Paper 26:** DSE (about +53 % GFLOPs in Paper 26, too heavy for the Raspberry Pi 5) and LSA (neck).

**NCNN note.** LayerNorm over the channels is written with mean/variance over the channel axis on a 4-D tensor.

**Assumptions** (not stated in Paper 26): adapter width 64, dropout 0.1 (inactive at inference), γ initialised to 1e-6
and γx to 1 (as in the original Mona adapter), identity / zero initialisation for a stable start.

In [ ]:
%%writefile msra_backbone.py
"""Paper 26 backbone for YOLO26n: C2PSA at layer 10 -> MSRA (C2PSA layout with 2 stacked Mona adapters instead of the PSA block)."""
import torch
import torch.nn as nn

from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.modules.block import C2PSA
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

MSRA_LAYER = 10
N_ADAPTERS = 2
ADAPTER_DIM = 64


class ChannelLayerNorm(nn.Module):
    """LayerNorm over the channel axis of a (B, C, H, W) tensor, written with 4-D ops."""

    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(1, c, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.eps = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        return (x - u) / torch.sqrt(s + self.eps) * self.weight + self.bias


class MonaAdapter(nn.Module):
    """Mona adapter: LN mix -> 1x1 down -> (DW3 + DW5 + DW7)/3 + id -> 1x1 mix + id -> GELU, dropout -> 1x1 up -> residual."""

    def __init__(self, c, dim=ADAPTER_DIM, p=0.1):
        super().__init__()
        self.norm = ChannelLayerNorm(c)
        self.gamma = nn.Parameter(torch.full((1, c, 1, 1), 1e-6))
        self.gammax = nn.Parameter(torch.ones(1, c, 1, 1))
        self.down = nn.Conv2d(c, dim, 1)
        self.dw3 = nn.Conv2d(dim, dim, 3, 1, 1, groups=dim)
        self.dw5 = nn.Conv2d(dim, dim, 5, 1, 2, groups=dim)
        self.dw7 = nn.Conv2d(dim, dim, 7, 1, 3, groups=dim)
        self.mix = nn.Conv2d(dim, dim, 1)
        self.act = nn.GELU()
        self.drop = nn.Dropout(p)
        self.up = nn.Conv2d(dim, c, 1)
        nn.init.zeros_(self.up.weight)  # adapter starts as identity
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        p1 = self.down(self.norm(x) * self.gamma + x * self.gammax)
        dm = (self.dw3(p1) + self.dw5(p1) + self.dw7(p1)) / 3.0 + p1
        do = dm + self.mix(dm)
        return x + self.up(self.drop(self.act(do)))


class MSRA(nn.Module):
    """C2PSA layout: cv1 -> split (Ra, Rb) -> Ra: 1x1 projection, Rb: n Mona adapters -> concat -> cv2."""

    def __init__(self, c2psa, n=N_ADAPTERS, dim=ADAPTER_DIM):
        super().__init__()
        self.c = c2psa.c
        self.cv1, self.cv2 = c2psa.cv1, c2psa.cv2  # pretrained 1x1 convs kept
        self.proj_a = nn.Conv2d(self.c, self.c, 1, bias=False)
        with torch.no_grad():
            self.proj_a.weight.copy_(torch.eye(self.c).view(self.c, self.c, 1, 1))  # starts as identity
        self.m = nn.Sequential(*(MonaAdapter(self.c, dim) for _ in range(n)))

    def forward(self, x):
        a, b = self.cv1(x).split((self.c, self.c), dim=1)
        return self.cv2(torch.cat((self.proj_a(a), self.m(b)), 1))


def add_msra(model, layer=MSRA_LAYER):
    """Replace the C2PSA at the given layer with MSRA (cv1/cv2 keep their pretrained weights). Returns the model."""
    old = model.model[layer]
    assert isinstance(old, C2PSA), f"layer {layer} is {type(old).__name__}, not C2PSA"
    new = MSRA(old).to(next(old.parameters()).device)
    new.i, new.f, new.type = old.i, old.f, "MSRA"  # keep the routing attributes used by the model's forward
    new.np = sum(p.numel() for p in new.parameters())
    model.model[layer] = new
    return model


class MSRATrainer(DetectionTrainer):
    """Loads the pretrained weights first, then swaps the C2PSA at layer 10 for MSRA."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DetectionModel(cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=False)
        if weights:
            model.load(weights)
        add_msra(model)
        if RANK in {-1, 0}:
            print(f"Paper 26 backbone: MSRA ({N_ADAPTERS} Mona adapters) at layer {MSRA_LAYER}")
        if verbose and RANK in {-1, 0}:
            model.info()
        return model


In [ ]:
import importlib, msra_backbone
importlib.reload(msra_backbone)
from msra_backbone import add_msra, MSRATrainer
from ultralytics.nn.tasks import DetectionModel, yaml_model_load
from ultralytics.utils.torch_utils import get_flops

def fused_params_gflops(model, imgsz=640):
    m = deepcopy(model).float().cpu().eval()
    m.fuse(verbose=False)
    return sum(p.numel() for p in m.parameters()) / 1e6, get_flops(m, imgsz)

# structure check (untrained): base vs Paper 26 backbone
base = DetectionModel(yaml_model_load("yolo26n.yaml"), nc=len(names), verbose=False).eval()
msra = add_msra(deepcopy(base)).eval()
with torch.no_grad():
    _ = msra(torch.zeros(1, 3, 640, 640))  # full forward check
    x = torch.randn(1, msra.model[10].c, 20, 20)
    print("Mona adapters start as identity:", all(torch.allclose(a(x), x, atol=1e-6) for a in msra.model[10].m))
print("layer 10:", type(msra.model[10]).__name__, "| branch width c:", msra.model[10].c, "| adapters:", len(msra.model[10].m))
for tag, m in (("base", base), ("Paper 26 backbone", msra)):
    pm, gf = fused_params_gflops(m)
    print(f"{tag:18s} Params {pm:.4f} M | GFLOPs {gf:.3f}")

## 4. Training

Only the Paper 26 backbone model is trained.

In [ ]:
from ultralytics import YOLO

EPOCHS  = 100
IMGSZ   = 640
BATCH   = 16
WORKERS = 2
SEED    = 0
PROJECT = str(WORK / "runs")

COMMON = dict(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, deterministic=True,
    pretrained="yolo26n.pt", device=0, workers=WORKERS, cache=False, patience=30, cos_lr=False,
    project=PROJECT, exist_ok=True, plots=True,
    # channels_last=False: ultralytics' MuSGD crashes on channels_last weights ("view size is not compatible").
    optimizer="MuSGD", lr0=0.01, momentum=0.9, channels_last=False,
)

# (row label in the table, run name)
VARIANTS = [
    ("MSRA Backbone (Paper 26)", "bb_msra"),
]

def best_path(run):
    return Path(PROJECT) / f"{run}_s{SEED}" / "weights" / "best.pt"

for label, run in VARIANTS:
    print(f"\n========== {label} ({run}) ==========")
    model = YOLO("yolo26n.yaml")
    model.train(trainer=MSRATrainer, **dict(COMMON, name=f"{run}_s{SEED}"))
    del model; gc.collect(); torch.cuda.empty_cache()

## 5. Test-set evaluation: Precision, Recall, F1-score, mAP@50, Params, GFLOPs

Params and GFLOPs are counted on the fused inference model, the same way as the baseline row.

In [ ]:
import msra_backbone  # must be imported before loading an MSRA checkpoint

rows = []
for label, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, train it first"); continue
    r = YOLO(str(w)).val(data=DATA_YAML, split="test", imgsz=IMGSZ, batch=16, device=0, plots=False, verbose=False)
    P, R = float(r.box.mp), float(r.box.mr)
    params, gflops = fused_params_gflops(YOLO(str(w)).model, IMGSZ)
    rows.append({"run": run, "Backbone Variant": label,
                 "Params (M)": round(params, 2), "GFLOPs": round(gflops, 2),
                 "Precision": round(P, 3), "Recall": round(R, 3),
                 "F1-score": round(2 * P * R / (P + R + 1e-9), 3),
                 "mAP@50": round(float(r.box.map50), 3)})
METRICS = pd.DataFrame(rows)
METRICS.drop(columns="run")

## 6. Package for Raspberry Pi 5 and Jetson

Creates `paper26_backbone_edge_bench.zip` in the **Output** tab with:

| Item | Used on |
|---|---|
| `bb_msra_ncnn_model/` (NCNN model, imgsz 640) | Raspberry Pi 5 |
| `bb_msra.pt` | Jetson |
| `bench_images/` (50 real test images, seed 0) | both |
| `msra_backbone.py` | Jetson (needed to load the MSRA `.pt`) |
| `pi5_bench.py`, `jetson_bench.py` | benchmark scripts |

Both scripts: batch 1, imgsz 640, cycle through the 50 real test images (loaded into memory first),
100 warm-up runs, then 1000 timed runs end-to-end (preprocessing + inference + postprocessing), and print the **median** ms and FPS.
Unzip into its own folder, e.g. `python -m zipfile -e paper26_backbone_edge_bench.zip paper26_backbone_edge_bench`.

In [ ]:
EDGE_IMGSZ = 640
N_BENCH_IMAGES = 50
EDGE = WORK / "paper26_backbone_edge_bench"
shutil.rmtree(EDGE, ignore_errors=True)
EDGE.mkdir()
shutil.copy("msra_backbone.py", EDGE / "msra_backbone.py")

# 50 real test images (fixed seed, so every model and device uses the same images)
test_imgs = sorted(p for p in (DATA_ROOT / "test" / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"})
random.Random(SEED).shuffle(test_imgs)
(EDGE / "bench_images").mkdir()
for p in test_imgs[:N_BENCH_IMAGES]:
    shutil.copy(p, EDGE / "bench_images" / p.name)
print(f"bench_images: {len(list((EDGE / 'bench_images').glob('*')))} real test images")

for _, run in VARIANTS:
    w = best_path(run)
    if not w.exists():
        print(f"{run}: best.pt not found, skipped"); continue
    shutil.copy(w, EDGE / f"{run}.pt")
    p = YOLO(str(w)).export(format="ncnn", imgsz=EDGE_IMGSZ, half=False)
    shutil.copytree(p, EDGE / f"{run}_ncnn_model")  # ultralytics needs the "_ncnn_model" suffix
    print(f"{run:12s} -> {run}.pt, {run}_ncnn_model/")

COMMON_BENCH = """
import glob, statistics, sys, time
import cv2

IMGSZ, WARMUP, RUNS = {imgsz}, 100, 1000
imgs = [cv2.imread(p) for p in sorted(glob.glob("bench_images/*"))]
imgs = [im for im in imgs if im is not None]
if not imgs:
    sys.exit("No images found in bench_images/. Run this script inside the unzipped folder.")
print(f"{{len(imgs)}} real test images, imgsz {{IMGSZ}}, batch 1, {{WARMUP}} warm-up + {{RUNS}} timed runs (median)")


def bench(model, **kw):
    for i in range(WARMUP):
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
    times = []
    for i in range(RUNS):
        t = time.perf_counter()
        model.predict(imgs[i % len(imgs)], imgsz=IMGSZ, verbose=False, **kw)
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)
""".format(imgsz=EDGE_IMGSZ)

PI5 = '''# Raspberry Pi 5 benchmark (NCNN). Run inside the unzipped folder:  python pi5_bench.py
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*_ncnn_model"))
if not models:
    sys.exit("No *_ncnn_model folders found.")
for path in models:
    ms = bench(YOLO(path, task="detect"))
    print(f"{path.replace('_ncnn_model', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

JETSON = '''# Jetson benchmark (PyTorch, CUDA FP16). Run inside the unzipped folder:  python jetson_bench.py
import sys
import torch

if not torch.cuda.is_available():
    sys.exit("WARNING: CUDA GPU not available. Install the NVIDIA Jetson PyTorch wheel for your JetPack, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, ".")
import msra_backbone  # needed to load the MSRA checkpoint
from ultralytics import YOLO
''' + COMMON_BENCH + '''
models = sorted(glob.glob("*.pt"))
if not models:
    sys.exit("No .pt files found.")
for path in models:
    ms = bench(YOLO(path, task="detect"), device=0, half=True)
    print(f"{path.replace('.pt', ''):12s}  median {ms:8.2f} ms   {1000 / ms:7.2f} FPS")
'''

(EDGE / "pi5_bench.py").write_text(PI5)
(EDGE / "jetson_bench.py").write_text(JETSON)
shutil.make_archive(str(WORK / "paper26_backbone_edge_bench"), "zip", EDGE)
print("\nDownload paper26_backbone_edge_bench.zip from the Output tab.")

### How to run on the devices

**Raspberry Pi 5** (same Pi and same conditions as the baseline measurement)
1. Activate the Python environment that has `ultralytics==8.4.137` and `ncnn`.
2. `echo performance | sudo tee /sys/devices/system/cpu/cpu*/scaling_governor`
3. Go into the unzipped folder and run `python pi5_bench.py`.
4. Run it 3 times with a 2–3 minute pause; use the middle value. Check `vcgencmd measure_temp` and `vcgencmd get_throttled` (`throttled=0x0` = OK).

**Jetson**
1. NVIDIA Jetson PyTorch + torchvision for your JetPack, then `ultralytics==8.4.137`.
2. `sudo nvpmodel -m 0` (check with `sudo nvpmodel -q`) and `sudo jetson_clocks`.
3. Go into the unzipped folder and run `python jetson_bench.py`. Run 3 times; use the middle value.

## 7. Final backbone table

The baseline row is already filled in. Type the median FPS of the Paper 26 backbone model
(and the baseline Jetson FPS when you have it), keep `None` for a missing value (shown as `--`), then run the cell.
Run section 5 first in the same session (it fills `METRICS`).

In [ ]:
# baseline values (identical YOLO26n model)
BASE_ROW = {"Backbone Variant": "Standard YOLO26n Backbone", "Params (M)": 2.38, "GFLOPs": 5.34,
            "Precision": 0.971, "Recall": 0.937, "F1-score": 0.953, "mAP@50": 0.958,
            "FPS (Rasp-5)": 13.38, "FPS (Jetson)": None}

FPS_PI5    = {"bb_msra": None}
FPS_JETSON = {"bb_msra": None}

table = METRICS.copy()
table["FPS (Rasp-5)"] = table["run"].map(lambda r: FPS_PI5.get(r))
table["FPS (Jetson)"] = table["run"].map(lambda r: FPS_JETSON.get(r))
COLS = ["Backbone Variant", "Params (M)", "GFLOPs", "Precision", "Recall", "F1-score", "mAP@50", "FPS (Rasp-5)", "FPS (Jetson)"]
table = pd.concat([pd.DataFrame([BASE_ROW]), table[COLS]], ignore_index=True)[COLS]
table = table.astype(object).where(table.notna(), "--")
table.style.hide(axis="index")

## 8. Zip all results

Run this last. It creates **`paper26_backbone_all_results.zip`** in the **Output** tab with:

| Folder in the zip | Contents |
|---|---|
| `paper26_backbone_edge_bench/` | NCNN + `.pt` model, 50 bench images, `msra_backbone.py`, `pi5_bench.py`, `jetson_bench.py` |
| `weights/` | `best.pt`, renamed `bb_msra_best.pt` |
| `runs/` | full training folder: `best.pt`, `last.pt`, `results.csv`, training curves, confusion matrix, PR curves |

In [ ]:
import zipfile

ALL_ZIP = WORK / "paper26_backbone_all_results.zip"
folders = [WORK / "paper26_backbone_edge_bench", WORK / "runs"]

with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        if not folder.exists():
            print(f"{folder.name}: not found, skipped"); continue
        n = 0
        for f in folder.rglob("*"):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
                n += 1
        print(f"{folder.name + '/':30s} {n} files")
    for _, run in VARIANTS:
        w = best_path(run)
        if w.exists():
            z.write(w, f"weights/{run}_best.pt")
            print(f"weights/{run}_best.pt")

print(f"\nDownload {ALL_ZIP.name} ({ALL_ZIP.stat().st_size / 1e6:.1f} MB) from the Output tab.")